I need to come up with parameters of what a good reconstruction

- The reconstruction is faithful to input.  
  - first lets do this, we want a dictionary learning which gives faithful representations
  - For now, i look at the representations
  - The loss is quite opaque in this i feel from dict learning
  - Lets try on sparse components
  - Although dict learning is very resistent to noise, i don't know how to compare good fits because the loss is always so small
- I'm able to tag the input with the reconstructed data.  
  - for now, we simply wanna tag the patches which were "caught" correctly
  - This is dependent on the fact that we get faithful reconstruction :)
- Generally, measuring faithfulness itself is slightly difficult lol


Its working reasonably well for (2,3), lets do it for the whole kernel activation space now.  


We have a problem though, the problem is basically contrib is scaled per pixel. Some pixels are astronomically useless (like borders).  
This is a problem of the saliency method for some reason, i don't know though.  
Do we need a minimal hard constraint? like using 0.01?  

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import os
import sys
import django

# Setup Django environment
# Adjust the path to point to the directory containing manage.py
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../../hiccup_ide")))
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "hiccup_ide.settings")
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
django.setup()

In [ ]:
from neural_data.models import *
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import random
import torch
from pt_to_api.utils import (
    show_single_channel_red_green_black as S,
    to_show_list as tsl,
    show_72,
    show_72_list,
    scatter_plot_1d,
    mk_rect_on_ax,
    get_receptive,
    otsu_threshold,
    explain_variance_with_pca,
)
from tqdm import tqdm
from pt_to_api.contribs.v1 import (
    show_input_patch_and_kernel_placement_for_poi_using_raw_params as SIP,
)
import seaborn as sns
import numpy as np
from sklearn.decomposition import MiniBatchDictionaryLearning, DictionaryLearning
from sklearn.preprocessing import Normalizer
from sklearn.metrics.pairwise import (
    pairwise_distances,
    cosine_similarity,
    cosine_distances,
)
from scipy.optimize import linear_sum_assignment
import pandas as pd

from neural_data.utils import (
    get_full_conv_kernel_at_coordinate,
    get_saliency_map_ids_and_patches,
    get_full_activations_of_layer,
)
from collections import defaultdict
import itertools

In [ ]:
KERNEL_COORDINATE = "layers.2.out_12"
INPUT_LAYER_NAME = "layers.1"
MODE = "light"
R = 2
C = 3

# Data gather

In [ ]:
len(SaliencyMap.objects.filter(coordinate=KERNEL_COORDINATE).first().data)

In [ ]:
# i should do otsu based thresholding and minimum value to 0.01 per point
kernel = get_full_conv_kernel_at_coordinate(KERNEL_COORDINATE)
vals = []
poi_by_data = {}
sm0 = SaliencyMap.objects.filter(coordinate=KERNEL_COORDINATE).first().data
MAX_R = len(sm0)
MAX_C = len(sm0[0])
MIN_POS_THRESH = 0.01
MIN_NEG_THRESH = -0.01


all_sms = SaliencyMap.objects.filter(coordinate=KERNEL_COORDINATE)

for i in range(0, MAX_R):
    for j in range(0, MAX_C):
        vals = [sm.data[i][j] for sm in all_sms]

        pvals = [v for v in vals if v > 0]
        pos_thresh = otsu_threshold(pvals)
        pos_thresh = max(pos_thresh, MIN_POS_THRESH)

        nvals = [v for v in vals if v < 0]
        neg_thresh = otsu_threshold(nvals)
        neg_thresh = min(neg_thresh, MIN_NEG_THRESH)


        print(f"### ({i}, {j})")
        poi_by_data[(i,j)] = get_saliency_map_ids_and_patches(
            (i,j), KERNEL_COORDINATE, INPUT_LAYER_NAME, pos_thresh, neg_thresh
        )

In [ ]:
poi_by_data[1,3][0]

In [ ]:
for r in range(MAX_R):
    for c in range(MAX_C):
        if len(poi_by_data[r,c][0]) > 0:
            print(r, c, len(poi_by_data[r,c][0]))

In [ ]:
poi_by_data[2,3][1][0].shape

In [ ]:
# get with lower thresholds
from neural_data.utils import get_all_saliency_map_ids_and_patches, get_saliency_map_ids_and_patches


# sm_ids, patches = get_saliency_map_ids_and_patches((R,C), KERNEL_COORDINATE, INPUT_LAYER_NAME, pos_thresh, neg_thresh)
# sm_ids, patches = get_all_saliency_map_ids_and_patches(
#     KERNEL_COORDINATE, INPUT_LAYER_NAME, pos_thresh, neg_thresh
# )
patches = []
for r in range(MAX_R):
    for c in range(MAX_C):
        patches.extend(poi_by_data[r,c][1])
pw = [p * kernel for p in patches]
lin_pw = np.array([p.reshape(-1) for p in pw])

# metrics and functions

In [ ]:
from typing import Literal


def _single_dict_learn(
    X, n_components, alg: Literal["minibatch", "basic"], seed=None, **kwargs
):
    clz = DictionaryLearning if alg == "basic" else MiniBatchDictionaryLearning
    dl = clz(n_components=n_components, random_state=seed, **kwargs)
    dl.fit(X)
    codes = dl.transform(X)
    error = np.mean((X - codes @ dl.components_) ** 2)
    new_comps = finetune_dictionary(X, dl.components_, codes)
    dl.components_ = new_comps
    return dl, codes, error


def finetune_dictionary(X, atoms, sparse_codes, lr=1e-4, n_iter=500, lam=1e-3):
    """
    X: (n_samples, n_features)
    atoms: (n_atoms, n_features)
    sparse_codes: (n_samples, n_atoms)
    """
    A = atoms.copy().astype(float)
    Z = sparse_codes.T  # (n_atoms, n_samples)

    for i in range(n_iter):
        residual = X - Z.T @ A      # (n_samples, n_features)
        # print(residual.sum())
        grad = -2 * Z @ residual    # (n_atoms, n_features)
        grad += 2 * lam * A         # L2 regularization
        A -= lr * grad

    return A

def do_dict_learn(
    inputs,
    n_components,
    alg: Literal["minibatch", "basic"] = "basic",
    n_init=5,
    **kwargs,
):
    # X = Normalizer("max").fit_transform(inputs)
    X = inputs.copy()
    kwargs = {
        "alpha": 1,
        "max_iter": 5000,
        "fit_algorithm": "cd",
        "positive_code": True,
        **kwargs,
    }
    if n_init == 1:
        dl, codes, error = _single_dict_learn(X, n_components, alg, None, **kwargs)
        return (dl, codes, X), [dl]

    best_dl, best_codes, best_error = None, None, np.inf
    all_dls = []
    for seed in range(n_init):
        dl, codes, error = _single_dict_learn(X, n_components, alg, seed, **kwargs)
        all_dls.append(dl)
        if error < best_error:
            best_dl, best_codes, best_error = dl, codes, error
    return (best_dl, best_codes, X), all_dls


In [ ]:
def get_sparse_codes_ratio_with_more_than_k_active_node(
    sparse_codes, k, alpha_for_code_below=0.05
):
    idxes = []
    for i in range(len(sparse_codes)):
        non_zero_codes = sparse_codes[i][sparse_codes[i] != 0]
        if len(non_zero_codes) == 0:
            continue
        non_zero_codes = np.abs(non_zero_codes)
        max_val_idx = np.argmax(non_zero_codes)
        thresh = non_zero_codes[max_val_idx] * alpha_for_code_below

        gt_thresh = []
        for j in range(len(non_zero_codes)):
            if j == max_val_idx:
                continue
            if non_zero_codes[j] > thresh:
                gt_thresh.append(non_zero_codes[j])
        if len(gt_thresh) > k:
            idxes.append(i)
    return len(idxes) / sparse_codes.shape[0]

def get_sparse_codes_ratio_with_more_than_1_active_node(
    sparse_codes, alpha_for_code_below=0.05
):
    return get_sparse_codes_ratio_with_more_than_k_active_node(sparse_codes, 1, alpha_for_code_below)

def plot_error_metrics(l2_dicts):
    df = pd.DataFrame.from_dict(l2_dicts, orient="index")
    metrics = ["max", "mean", "50", "75", "90", "95"]
    fig, axes = plt.subplots(2, 3, figsize=(12, 6))

    for ax, metric in zip(axes.flatten(), metrics):
        ax.plot(df.index, df[metric])
        ax.set_title(metric)
    plt.show()


def get_loss(dl, sparse_codes, X):
    reconstructed = sparse_codes @ dl.components_
    diff = X - reconstructed
    err_sq = np.linalg.norm(diff, axis=1)
    return err_sq


def get_metrics(dl, sparse_codes, X):
    err_sq = get_loss(dl, sparse_codes, X)
    _p = np.percentile
    metrics = {
        "l2": {
            "max": np.max(err_sq),
            "mean": np.mean(err_sq),
            "50": _p(err_sq, 50),
            "75": _p(err_sq, 75),
            "90": _p(err_sq, 90),
            "95": _p(err_sq, 95),
        },
        "err": err_sq,
    }
    return metrics


def show_dl_comps(comps):
    c = list(itertools.chain.from_iterable([tsl(c.reshape(8, 3, 3)) for c in comps]))
    S(c, (20, 15), 8, mode=MODE)
    plt.show()


def get_sparsity_in_rows(sparse_codes):
    # < 0.2 is good, < 0.1 is quite nice
    each_row_sparsity = (sparse_codes != 0).sum(axis=1) / sparse_codes.shape[1]
    _p = np.percentile
    return {
        "max": np.max(each_row_sparsity),
        "mean": np.mean(each_row_sparsity),
        "50": _p(each_row_sparsity, 50),
        "75": _p(each_row_sparsity, 75),
        "90": _p(each_row_sparsity, 90),
        "95": _p(each_row_sparsity, 95),
    }

In [ ]:
def make_sparse(patch, num_std=1):
    mean, std = np.mean(patch), np.std(patch)
    thresh = mean + num_std * std

    sparse = patch.copy()
    sparse[np.abs(sparse) < thresh] = 0
    return sparse

# standard dict learn

In [ ]:
sparse_pw = lin_pw
(dl, sparse_codes, X), all_dls = do_dict_learn(
    sparse_pw,
    10,
    "minibatch",
    alpha=1.2,
    max_iter=10000,
    transform_max_iter=2000,
    transform_algorithm="lasso_cd",
)
atoms = finetune_dictionary(sparse_pw, dl.components_, sparse_codes)
recons = sparse_codes @ atoms


print("recons, normalized")
show_72(recons[0], mode=MODE)
print("original, normalized")
show_72(sparse_pw[0], mode=MODE)

# tis no good without normalisation, loss lies
print("recons and original, together, same scale")
show_72_list([recons[0], sparse_pw[0]], mode=MODE)

# SAE

## Vanilla

In [ ]:
import torch.nn as nn
import torch.optim as optim
 
 
class VanillaSparseAutoencoder(nn.Module):
    def __init__(self, input_dim, n_components):
        super().__init__()
        self.encoder = nn.Linear(input_dim, n_components)
        self.decoder = nn.Linear(n_components, input_dim, bias=False)
        # normalize decoder columns to unit norm (same as sklearn)
        self._normalize_decoder()
 
    def _normalize_decoder(self):
        with torch.no_grad():
            norms = self.decoder.weight.norm(dim=0, keepdim=True).clamp(min=1.0)
            self.decoder.weight.div_(norms)
 
    def forward(self, x):
        codes = torch.relu(self.encoder(x))
        recon = self.decoder(codes)
        return recon, codes
 
 
def train_sae(X, n_components, alpha=0.1, beta=0.1, lr=1e-3, epochs=2000, batch_size=256, weights_init_sigma=None):
    """
    X: numpy array (n_samples, input_dim)
    n_components: number of dictionary atoms
    alpha: sparsity penalty weight
    """
    X_t = torch.tensor(X, dtype=torch.float32)
    n_samples, input_dim = X_t.shape
 
    model = VanillaSparseAutoencoder(input_dim, n_components)
    if weights_init_sigma is None:
        torch.nn.init.normal(model.decoder.weights, 0, weights_init_sigma)
    optimizer = optim.Adam(model.parameters(), lr=lr)
 
    for epoch in range(epochs):
        # shuffle
        idx = torch.randperm(n_samples)
        X_t = X_t[idx]
 
        epoch_loss = 0
        for i in range(0, n_samples, batch_size):
            batch = X_t[i:i+batch_size]
            recon, codes = model(batch)
 
            recon_loss = ((batch - recon) ** 2).mean()
            sparsity_loss = alpha * codes.abs().mean()
            loss = recon_loss + sparsity_loss
 
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            model._normalize_decoder()
 
            epoch_loss += loss.item()
 
        if epoch % 200 == 0:
            print(f"epoch {epoch:4d} | recon_loss {recon_loss:.4f} sparse_loss {sparsity_loss:.4f}")
 
    # final codes and reconstruction
    with torch.no_grad():
        recon, codes = model(X_t)
 
    return (
        model,
        codes.numpy(),
        model.decoder.weight.T.detach().numpy(),  # (n_components, input_dim)
        recon.numpy(),
    )

In [ ]:
sparse_pw = lin_pw
model, sparse_codes, components, recons = train_sae(sparse_pw, 10)

In [ ]:
i = 6
# SAE has better performance than dict learning i think
print("recons, normalized")
show_72(recons[i], mode=MODE)
print("original, normalized")
show_72(lin_pw[i], mode=MODE)

# tis no good without normalisation, loss lies
print("recons and original, together, same scale")
show_72_list([recons[i], lin_pw[i]], mode=MODE)

## draw some functions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(-10, 10, 1000)
lam = 5

# y = 1 - np.exp(-np.abs(x) / lam)
y = 1 - np.exp(-np.square(x) / lam)

plt.plot(x, y)
plt.xlabel('x')
plt.ylabel('1 - exp(-|x| / λ)')
plt.title(f'λ = {lam}')
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(-5, 5, 1000)

def sigmoid(x, c=0):
    return 1 / (1 + np.exp(-(x - c)))

# symmetric version for your use case
def sigmoid_abs(x, c=0):
    return 1 / (1 + np.exp(-(np.abs(x) - c)))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# vary c
for c in [0, 1, 2, 3]:
    axes[0].plot(x, sigmoid_abs(x, c), label=f'c={c}')
axes[0].set_title('sigmoid(|x| - c), varying c')
axes[0].legend()
axes[0].grid(True)

# compare with exponential
lam = 1.0
axes[1].plot(x, sigmoid_abs(x, c=1), label='sigmoid(|x| - 1)')
axes[1].plot(x, 1 - np.exp(-np.abs(x) / lam), label='1 - exp(-|x| / λ=1)', linestyle='--')
axes[1].set_title('sigmoid vs exponential')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(-5, 5, 1000)

def soft_abs(x, eps=1.0):
    return np.sqrt(x**2 + eps)
    # return x**2

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# vary eps
for eps in [0.01, 0.1, 0.5, 1.0]:
    axes[0].plot(x, soft_abs(x, eps), label=f'ε={eps}')
axes[0].plot(x, np.abs(x), label='|x|', linestyle='--', color='black')
axes[0].set_title('soft abs: sqrt(x² + ε)')
axes[0].legend()
axes[0].grid(True)

# plug into sigmoid
def sigmoid_softabs(x, c=1, eps=0.1):
    return 1 / (1 + np.exp(-(soft_abs(x, eps) - c)))

for eps in [0.01, 0.1, 0.5, 1.0]:
    axes[1].plot(x, sigmoid_softabs(x, c=0.1, eps=eps), label=f'ε={eps}')
axes[1].set_title('sigmoid(softabs(x) - c)')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(-5, 5, 1000)

def soft_abs(x, eps=0.01):
    return np.sqrt(x**2 + eps)

def tanh_softabs(x, lam=1.0, eps=0.01):
    return np.tanh(soft_abs(x, eps) / lam)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# vary lam
for lam in [0.5, 1.0, 2.0, 3.0, 10.0]:
    axes[0].plot(x, tanh_softabs(x, lam=lam), label=f'λ={lam}')
axes[0].set_title('tanh(softabs(x) / λ), varying λ')
axes[0].legend()
axes[0].grid(True)

# compare all candidates
axes[1].plot(x, tanh_softabs(x, lam=1.0), label='tanh(softabs(x))')
axes[1].plot(x, 1 - np.exp(-np.abs(x) / 1.0), label='1 - exp(-|x| / λ=1)', linestyle='--')
axes[1].set_title('tanh vs exponential')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## L2 on weight without normalization

Interesting, if the number of components is less, the model is driving down the sparse loss and l2 loss.  
What can be the reason for this?  

If there are not enough components, it might not be able to correctly make a good representation, 0 might be the best representation.  
Nice nice. Its an indicator anyways.  

In [ ]:
import torch.nn as nn
import torch.optim as optim
 
 
class SaeWeightsL2(nn.Module):
    def __init__(self, input_dim, n_components):
        super().__init__()
        self.encoder = nn.Linear(input_dim, n_components)
        self.decoder = nn.Linear(n_components, input_dim, bias=False)
 
    def forward(self, x):
        codes = torch.relu(self.encoder(x))
        recon = self.decoder(codes)
        return recon, codes
 
 
def train_sae_weights_l2(X, n_components, alpha=0.1, beta=0.1, lr=1e-3, epochs=2000, batch_size=256, weights_init_sigma=None):
    """
    X: numpy array (n_samples, input_dim)
    n_components: number of dictionary atoms
    alpha: sparsity penalty weight
    """
    X_t = torch.tensor(X, dtype=torch.float32)
    n_samples, input_dim = X_t.shape
 
    model = SaeWeightsL2(input_dim, n_components)
    if weights_init_sigma is not None:
        nn.init.normal_(model.decoder.weight, mean=0, std=weights_init_sigma)

    optimizer = optim.Adam(model.parameters(), lr=lr)
 
    for epoch in range(epochs):
        # shuffle
        idx = torch.randperm(n_samples)
        X_t = X_t[idx]
 
        epoch_loss = 0
        for i in range(0, n_samples, batch_size):
            batch = X_t[i:i+batch_size]
            recon, codes = model(batch)
 
            recon_loss = ((batch - recon) ** 2).mean()
            sparsity_loss = alpha * codes.abs().mean()
            l2_loss = beta * (model.decoder.weight ** 2).mean()
            loss = recon_loss + sparsity_loss + l2_loss
 
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
 
            epoch_loss += loss.item()
 
        if epoch % 200 == 0:
            print(f"epoch {epoch:4d} | recon_loss {recon_loss:.4f} sparse_loss {sparsity_loss:.4f}, l2_loss {l2_loss:.4f}")
 
    # final codes and reconstruction
    with torch.no_grad():
        recon, codes = model(X_t)
 
    return (
        model,
        codes.numpy(),
        model.decoder.weight.T.detach().numpy(),  # (n_components, input_dim)
        recon.numpy(),
    )

In [ ]:
# L2 on weights does weird shit
model, codes, decoder, recons = train_sae_weights_l2(lin_pw, 12)

In [ ]:
i = 6
# SAE has better performance than dict learning i think
print("recons, normalized")
show_72(recons[i], mode=MODE)
print("original, normalized")
show_72(lin_pw[i], mode=MODE)

# tis no good without normalisation, loss lies
print("recons and original, together, same scale")
show_72_list([recons[i], lin_pw[i]], mode=MODE)

## Normalized weights and inputs

I dont see much evidence of getting better with this also

In [ ]:
SIGMA = 0.5

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
lin_pw_scaled = scaler.fit_transform(lin_pw) * SIGMA

In [ ]:
show_72_list([lin_pw[0], lin_pw_scaled[0]], mode=MODE)

In [ ]:
model, codes, decoder, recons = train_sae(lin_pw_scaled, 20, weights_init_sigma=SIGMA, alpha=0.5)

In [ ]:
codes[10]

In [ ]:
i = 4
# SAE has better performance than dict learning i think
scaled_back_recons = scaler.inverse_transform(recons / SIGMA)
r = scaled_back_recons[i]
print("recons, normalized")
show_72(r, mode=MODE)
print("original, normalized")
show_72(lin_pw[i], mode=MODE)

# tis no good without normalisation, loss lies
print("recons and original, together, same scale")
show_72_list([r, lin_pw[i]], mode=MODE)

## Sparse mask SAE

Lets see if this gives anything useful

In [ ]:
import torch.nn as nn
import torch.optim as optim
 
 
class SparseAutoencoderWithSparseMask(nn.Module):
    def __init__(self, input_dim, n_components, clipper_denominator):
        super().__init__()
        self.e1 = nn.Linear(input_dim, n_components)
        self.e2 = nn.Linear(input_dim, n_components)
        # self.clipper_denominator = clipper_denominator

        self.decoder = nn.Linear(n_components, input_dim, bias=False)
 
    def forward(self, x):
        # y = 1 - np.exp(-np.square(x) / lam)
        codes = torch.relu(self.e1(x))
        sparse_coeffs = torch.relu(self.e2(x))

        # clipped_sparse_coeffs = 1 - torch.exp(-torch.square(sparse_coeffs) / self.clipper_denominator)
        # clipped_sparse_coeffs = torch.exp(-torch.square(sparse_coeffs) / self.clipper_denominator)

        multipliers = sparse_coeffs * codes
        recon = self.decoder(multipliers)

        return recon, codes, sparse_coeffs, multipliers
 
 
def train_sparse_mask_sae(X, n_components, alpha=0.1, beta=0.1, gamma=0.1, clipper_denominator=1, lr=1e-3, epochs=2000, batch_size=256):
    """
    X: numpy array (n_samples, input_dim)
    n_components: number of dictionary atoms
    alpha: sparsity penalty weight
    """
    X_t = torch.tensor(X, dtype=torch.float32)
    n_samples, input_dim = X_t.shape
 
    model = SparseAutoencoderWithSparseMask(input_dim, n_components, clipper_denominator)
    optimizer = optim.Adam(model.parameters(), lr=lr)
 
    for epoch in range(epochs):
        # shuffle
        idx = torch.randperm(n_samples)
        X_t = X_t[idx]
 
        epoch_loss = 0
        for i in range(0, n_samples, batch_size):
            batch = X_t[i:i+batch_size]
            recon, codes, sparse_coeffs, _ = model(batch)
 
            recon_loss = ((batch - recon) ** 2).mean()
            sparsity_loss = alpha * sparse_coeffs.abs().mean()
            loss = recon_loss + sparsity_loss
 
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
 
            epoch_loss += loss.item()
 
        if epoch % 200 == 0:
            print(f"epoch {epoch:4d} | recon_loss {recon_loss:.4f} sparse_loss {sparsity_loss:.4f}")
            # print(f"epoch {epoch:4d} | recon_loss {recon_loss:.4f} weights_l2 {l2_loss:.4f} sparse_loss {sparsity_loss:.4f} codes_loss {codes_loss:.4f}")
 
    # final codes and reconstruction
    with torch.no_grad():
        recon, codes, _, multipliers = model(X_t)
 
    return (
        model,
        multipliers.numpy(),
        model.decoder.weight.T.detach().numpy(),  # (n_components, input_dim)
        recon.numpy(),
    )

In [ ]:
# sparse_pw = make_sparse(lin_pw)
sparse_pw = lin_pw
# current_mean_norm = np.linalg.norm(sparse_pw, axis=-1).mean()
# normalized_sparse_pw = sparse_pw / current_mean_norm  # mean norm = 1

model, sparse_codes, components, recons = train_sparse_mask_sae(sparse_pw, 10)

In [ ]:
i = 10
# SAE has better performance than dict learning i think
print("recons, normalized")
show_72(recons[i], mode=MODE)
print("original, normalized")
show_72(lin_pw[i], mode=MODE)

# tis no good without normalisation, loss lies
print("recons and original, together, same scale")
show_72_list([recons[i], lin_pw[i]], mode=MODE)

# Dictionary learning compare

Technically this is also not bad, if the scale was okay.   
Generally, all of them have a problem with scale. It's the L1 killing scale.  

There is something about dictionary learning's algorithm, it does weights first, then scalar codes, then weights, then scalar codes. and ends up being quite stable (across different seeds). Compared to SAE which gives quite different reconstructions every time.  

Why?


I'm actually back at dictionary learning for some reason. There are two options now.
- Fix the shrinkage somehow in vanilla dictionary learning
- Fix seed stability in SAE (much harder?)

Coordinate descent does one atom at a time, which sounds promising.   
This is getting harder than i thought it would be, the obvious things are already done lol.  


Lets assume that the dictionary atoms are good now. we can freeze them. or maybe, we can freeze the sparse codes and update the dict atoms?  Lets try that.  
How would i want to do that? the simplest is basic gradient descent i guess.  

Well, its pretty much working now, basically removing noise was important lol.   

In [ ]:
# i had to use a very low alpha in this case, this is not very nice when i need to do 
# bulk learning. but for now, we do it for all manually atleast, its not a lot of work for mnist
# now there is also the problem of checking the top losses hmmm
# i would need a good birds eye view of what is happening in the dict
(dl, sparse_codes, X), all_dls = do_dict_learn(
    lin_pw,
    30,
    "minibatch",
    alpha=0.3,
    max_iter=10000,
    transform_max_iter=2000,
    transform_algorithm="lasso_cd",
    n_init=1,
)
recons = sparse_codes @ dl.components_

In [ ]:
i = 4
print(sparse_codes[i])

# tis no good without normalisation, loss lies
print("recons and original, together, same scale")
show_72_list([recons[i], lin_pw[i]], mode=MODE)

In [ ]:
for j, s in enumerate(sparse_codes[i]):
    if s != 0:
        show_72(dl.components_[j], mode=MODE)

In [ ]:
# the number of stuff we need to learn is significantly higher now
# lmaos
def sweep_over_n_components(inputs):
    l2_dicts = {}
    sparsity_per_row = {}
    distance_between_seeds = {}
    comps_gt_1_per_row_ratio = [0]  # placeholder for 0
    for comp in tqdm(range(1, 60, 5)):
        (dl, sparse_codes, X), all_dls = do_dict_learn(
            inputs,
            comp,
            "minibatch",
            alpha=0.3,
            max_iter=20000,
            transform_algorithm="lasso_cd",
            n_init=1,
        )
        error_metrics = get_metrics(dl, sparse_codes, X)
        sparsity_per_row[comp] = get_sparsity_in_rows(sparse_codes)
        l2_dicts[comp] = error_metrics["l2"]
        comps_gt_1_per_row_ratio.append(
            get_sparse_codes_ratio_with_more_than_1_active_node(sparse_codes)
        )

    print("n_components v/s loss")
    plot_error_metrics(l2_dicts)

    print("n_components v/s sparsity")
    plot_error_metrics(sparsity_per_row)

    plt.plot(comps_gt_1_per_row_ratio)
    plt.show()

In [ ]:
sweep_over_n_components(lin_pw)

In [ ]:
sparse_pw = lin_pw
(dl, sparse_codes, X), all_dls = do_dict_learn(
    sparse_pw,
    45,
    "minibatch",
    alpha=1.,
    max_iter=10_000,
    transform_max_iter=10_000,
    transform_algorithm="lasso_cd",
    n_init=2,
)
atoms = finetune_dictionary(sparse_pw, dl.components_, sparse_codes)
recons = sparse_codes @ atoms

In [ ]:
print("recons, normalized")
show_72(recons[0], mode=MODE)
print("original, normalized")
show_72(sparse_pw[0], mode=MODE)

# tis no good without normalisation, loss lies
print("recons and original, together, same scale")
show_72_list([recons[0], sparse_pw[0]], mode=MODE)

In [ ]:
loss_vec = get_loss(dl, sparse_codes, lin_pw)
loss_idxs = np.array(list(reversed(np.argsort(loss_vec))))

In [ ]:
i = loss_idxs[210]
# print("recons, normalized")
# show_72(recons[i], mode=MODE)
# print("original, normalized")
# show_72(lin_pw[i], mode=MODE)

# tis no good without normalisation, loss lies
print("recons and original, together, same scale")
show_72_list([recons[i], lin_pw[i]], mode=MODE)

Its quite hard to gauge how well the dictionary is doing.  

Two things might still be worth trying:
- scale the inputs per sample
- increase iters maybe


Before that, something more important is to get a gauge of how well dict learning performs for a given set of parameters and datasets:  
- how many examples are 0ed out?
- how many atoms are too similar? 
- see the distribution of usage.
- seed stability.
- reconstruction loss

The list is quite big, but is not generally sufficient.  
But its okay i guess, i would need to do manual inspection in general it seems.  

In [ ]:
get_sparse_codes_ratio_with_more_than_k_active_node(sparse_codes, 2)

In [ ]:
def get_sparse_components(sparse_codes, dl_components, index):
    codes = sparse_codes[index]
    return [codes[i] * dl_components[i] for i in range(len(codes)) if codes[i] != 0]

In [ ]:
import gc

gc.collect()

In [ ]:
show_72(sparse_pw[0])

In [ ]:
import numpy as np
from numpy.linalg import lstsq

def relaxed_lasso_codes(X, dictionary, lasso_codes):
    """
    Given lasso codes (sparse, shrunk), re-fit least squares on the
    active atoms only. Returns unbiased sparse codes.
    
    Parameters
    ----------
    X : array (n_samples, n_features)
    dictionary : array (n_components, n_features)  — dl.components_
    lasso_codes : array (n_samples, n_components)  — from dl.transform(X)
    
    Returns
    -------
    relaxed_codes : array (n_samples, n_components)
    """
    relaxed_codes = np.zeros_like(lasso_codes)

    for i, (x, code) in enumerate(zip(X, lasso_codes)):
        active = np.where(code != 0)[0]
        if len(active) == 0:
            continue
        active_atoms = dictionary[active]          # (n_active, n_features)
        coeffs, _, _, _ = lstsq(active_atoms.T, x, rcond=None)
        relaxed_codes[i, active] = coeffs

    return relaxed_codes

In [ ]:
(dl, sparse_codes, X), all_dls = do_dict_learn(
    sparse_pw,
    11,
    "minibatch",
    alpha=1.2,
    max_iter=10000,
    transform_max_iter=2000,
    transform_algorithm="lasso_cd",
)
recons = sparse_codes @ dl.components_

print(
    "ratio of sparse codes which have > 1 components",
    get_sparse_codes_ratio_with_more_than_1_active_node(sparse_codes),
)
print(
    "sparsity inside rows (number of non-zero / number of eles in a row)\n",
    get_sparsity_in_rows(sparse_codes),
)
print("loss histogram across exmaples")
plt.hist(get_loss(dl, sparse_codes, X))
plt.show()


print("count of the number of times an atom is used")
activation_freq = (sparse_codes != 0).sum(axis=0)
plt.figure(figsize=(12, 4))
plt.bar(range(len(activation_freq)), np.sort(activation_freq)[::-1])
plt.xlabel("Atom rank")
plt.ylabel("Activation count")
plt.title("Atom activation frequency")
plt.show()

In [ ]:
relaxed_codes = relaxed_lasso_codes(X, dl.components_, sparse_codes)

In [ ]:
relaxed_recons = relaxed_codes @ dl.components_

In [ ]:
def show_normalized(x, recons):
    norm = Normalizer("max")
    show_72_list(
        [
            norm.fit_transform(recons.reshape(1, -1)),
            norm.fit_transform(x.reshape(1, -1)),
        ],
        mode=MODE,
    )
    plt.show()

In [ ]:
show_72_list([X[3], relaxed_recons[3]])

In [ ]:
losses = get_loss(dl, sparse_codes, X)
sorted_loss_idxs = np.argsort(losses)

In [ ]:
# very weird that these got into the main code
X[i], sparse_pw[i]

In [ ]:
# huh, it seems there are empty values in this
# lol
# interestingly, this is the highest loss
# which is not very bad
# a recurring problem is that the final things are much smaller in magnitude
# compared to what we want
# should i change my normalizing?
i = sorted_loss_idxs[-1]
show_72_list([X[i], recons[i]])

In [ ]:
# these are definitely not pure components, have a lot of redundancy
# im also not able to get perfect sparsity
# print(sparse_codes)
# show_72_list([norm.fit_transform(recons[0].reshape(1,-1)), norm.fit_transform(X[0].reshape(1,-1))], mode=MODE)
print("recons vs orig normalized")
# show_normalized(recons[0], X[0])
show_72(recons[3])
show_72(X[3])

print("sparse components")
# there was only one component lol
show_72_list(get_sparse_components(sparse_codes, dl.components_, 0), mode=MODE)

In [ ]:
def patch_from_recons(recons, kernel):
    # simply pointwise divide by kernel
    k = kernel.reshape(-1)
    recons = recons.reshape(-1)
    # patch = recons / k
    patch = np.where(k == 0, np.nan, recons / k)
    return patch.reshape(8, 3, 3)

In [ ]:
patches[0].shape

In [ ]:
S(tsl(patches[0]), (20, 8), 8, mode=MODE)
S(tsl(patch_from_recons(recons[0], kernel)), (20, 8), 8, mode=MODE)
S(tsl(kernel), (20, 8), 8, mode=MODE)
plt.show()

The main job is somehow being able to interpret these results.  
Since the reconstructed array has very less noise, we want to create a metric on how similar each is to the other.  
we can do this using cosine distances, or euclidean, lets see

In [ ]:
distances = cosine_distances(recons)
idx = np.argmin(distances[0][1:]).item()
idx, distances[0][idx]

The reconstruction is not very bad though, its now useful to see the similarities between the two by comparing the input images

In [ ]:
input_acts_a = get_full_activations_of_layer(
    "layers.1", Input.objects.get(alias=sm_ids[0]["input"])
)
input_acts_b = get_full_activations_of_layer(
    "layers.1", Input.objects.get(alias=sm_ids[idx]["input"])
)

In [ ]:
def get_rect(y, x, ksize=3, stride=2, padding=1):
    (y0, x0), (y1, x1) = get_receptive(y, x, ksize, stride, padding)
    return (y0, x0, y1 - y0, x1 - x0)

In [ ]:
axes = S(tsl(input_acts_a), (15, 5), ncols=8, mode=MODE)
for ax in axes:
    mk_rect_on_ax(ax, *get_rect(R, C))
plt.show()
axes = S(tsl(input_acts_b), (15, 5), ncols=8, mode=MODE)
for ax in axes:
    mk_rect_on_ax(ax, *get_rect(R, C))
plt.show()

In [ ]:
S(tsl(patch_from_recons(recons[0], kernel)), (20, 8), 8, mode=MODE)
plt.show()

In [ ]:
def patch_from_recons(recons, kernel):
    # simply pointwise divide by kernel
    k = kernel.reshape(-1)
    recons = recons.reshape(-1)
    # patch = recons / k
    patch = np.where(k == 0, np.nan, recons / k)
    return patch.reshape(8, 3, 3)

Reconstruction is bad for smaller components for sure, not sure if that is a good thing or bad for now.  

# Check what pattern a component is finding

- The first and the most interesting is seeing if patterns are coherent.
  - That is, i can find what meaning came out
  - The easiest is to first find an input where a single component fired alone.  
    - In this case, i would like to see the pattern the component caught
    - basically make rects around all the vector components where the component was non-zero
- We do this across input activations

In [ ]:
def find_pure_patches(codes, k, threshold=1e-3):
    other_cols = np.concatenate([codes[:, :k], codes[:, k + 1 :]], axis=1)
    mask = (np.abs(codes[:, k]) > threshold) & (
        np.all(np.abs(other_cols) < threshold, axis=1)
    )
    return np.where(mask)[0]


def find_high_contribution_patches(codes, k, percentile=75):
    contributions = np.abs(codes[:, k])
    threshold = np.percentile(contributions[contributions > 0], percentile)
    return np.where(contributions >= threshold)[0]


def find_patches_for_component(codes, k, pure_threshold=1e-3, high_percentile=75):
    pure = find_pure_patches(codes, k, threshold=pure_threshold)
    high = find_high_contribution_patches(codes, k, percentile=high_percentile)

    high_only = high[~np.isin(high, pure)]
    high_only_sorted = high_only[np.argsort(np.abs(codes[high_only, k]))[::-1]]

    return np.concatenate([pure, high_only_sorted])

In [ ]:
def draw_patches_with_rects_on_coords(input_acts, recons_3d, non_zero_idxs):
    print("input activations")
    axes = S(tsl(input_acts), (20, 8), 8, mode=MODE)
    for ax_idx, ax in enumerate(axes):
        for non_zero_i in non_zero_idxs:
            if non_zero_i[0] == ax_idx:
                _, y_rel, x_rel = non_zero_i
                (y0, x0), _ = get_receptive(R, C)
                pw_val = recons_3d[*non_zero_i]
                color = "r" if pw_val < 0 else "black"
                mk_rect_on_ax(ax, y0 + y_rel, x0 + x_rel, 1, 1, color)
    plt.show()


def summarize_pattern_of_component(comp_num, sm_ids, sparse_codes, dl, kernel):
    if sparse_codes[:, comp_num].sum() == 0:
        print("dead component, skipping")
        return
    idxes = find_patches_for_component(sparse_codes, comp_num)
    if len(idxes) == 0:
        print(f"could not find any contribution of component {comp_num}, its redundant")
        return

    i = idxes[0]
    for j, i in enumerate(idxes[:5]):
        recons_i = sparse_codes[i] @ dl.components_
        recons_3d = recons_i.reshape(8, 3, 3)
        non_zero_idxs = np.argwhere(np.abs(recons_3d) > 0.005)

        input_acts = get_full_activations_of_layer(
            "layers.1", Input.objects.get(alias=sm_ids[i]["input"])
        )
        patch_i = patch_from_recons(recons_i, kernel)

        if j == 0:
            print("first patch")
            print("reconstructed pointwise")
            show_72(recons_i, mode=MODE)
            plt.show()
            print("reconstructed patch")
            S(tsl(patch_i), (20, 8), 8, mode=MODE)
            plt.show()
            print("kernel")
            S(tsl(kernel), (20, 8), 8, mode=MODE)
            plt.show()

        print("##### input activations ######")
        draw_patches_with_rects_on_coords(input_acts, recons_3d, non_zero_idxs)

    # print("activations for multiple inputs")
    # for j, i in enumerate(idxes[:5]):
    #     recons_i = (sparse_codes[i] @ dl.components_)
    #     # recons_3d = recons_i.reshape(8,3,3)
    #     # nonzero_spatial = np.any(recons_3d != 0, axis=0)  # (3, 3) bool mask
    #     # nonzero_rc = np.argwhere(nonzero_spatial)  # list of (r, c) in patch space
    #     non_zero_idxs = np.argwhere(np.abs(recons_i.reshape(8,3,3)) > 0.005)

In [ ]:
summarize_pattern_of_component(3, sm_ids, sparse_codes, dl, kernel)

In [ ]:
summarize_pattern_of_component(3, sm_ids, sparse_codes, dl, kernel)

In [ ]:
# def get_rect(y, x, ksize=3, stride=2, padding=1):
#     (y0,x0), (y1,x1) = get_receptive(y, x, ksize, stride, padding)
#     return (y0, x0, y1-y0, x1-x0)
axes = S(tsl(input_acts_0), (15, 5), ncols=8, mode=MODE)
for ax in axes:
    mk_rect_on_ax(ax, *get_rect(R, C))
plt.show()
axes = S(tsl(input_acts_21), (15, 5), ncols=8, mode=MODE)
for ax in axes:
    mk_rect_on_ax(ax, *get_rect(R, C))
plt.show()

In [ ]:
find_pure_patches(sparse_codes, 7, 1e-5)

In [ ]:
sparse_codes[130]

In [ ]:
sparse_codes.shape

In [ ]:
# i.